# TrustLens Day 2 — Deepfake Classifier (EfficientNet-B0) on **Kaggle**

Transfer-learn EfficientNet-B0 on **GenImage**, holding out **one generator family** as out-of-distribution (OOD). Headline result: the **OOD accuracy drop**.

### Kaggle setup (do this first, right sidebar)
1. **Settings → Accelerator → GPU** (T4 x2 or P100).
2. **Settings → Internet → On** (needed for pip + pretrained weights).
3. **+ Add Input** → search `vtphatt2 GenImage` → attach **2+ generator datasets**, e.g.:
   - `GenImage-BigGAN`  (held out as OOD)
   - `genimage-stable-diffusion-v1-4`  (training)
   - optionally `GenImage-wukong`, `GenImage-ADM`, … (more training families)

Each attached dataset mounts read-only at `/kaggle/input/<slug>/` — **no download, no disk hit**, so you use the full subsets. Then run top-to-bottom.

## 0. GPU check

In [ ]:
!nvidia-smi -L
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Install TrustLens
torch/torchvision are preinstalled on Kaggle; we add mlflow and the package (from GitHub — make sure **Internet is On**).

In [ ]:
!pip -q install mlflow
!pip -q install git+https://github.com/umerjavaidkh/TrustLens.git
import trustlens.training.train, trustlens.models.splits
print("TrustLens Day-2 modules loaded OK")

## 2. Index the attached GenImage datasets
Each attached dataset under `/kaggle/input` is treated as one **generator family**. Unrelated attached datasets (no ai/nature folders) are ignored.

In [ ]:
from trustlens.models.splits import index_kaggle_inputs, list_generators
records = index_kaggle_inputs("/kaggle/input")
print(f"Indexed {len(records)} labelled images")
GENERATORS = list_generators(records)
print("GENERATORS =", GENERATORS)
assert len(GENERATORS) >= 2, "Attach >=2 GenImage generator datasets via + Add Input."

def pick_ood(gens, prefer="biggan"):
    for g in gens:
        if prefer in g.lower():
            return g
    return gens[0]

OOD_GEN = pick_ood(GENERATORS)
print("OOD_GENERATOR =", OOD_GEN, " (edit to hold out a different family)")

## 3. Config

In [ ]:
from trustlens.training.train import TrainConfig
cfg = TrainConfig(
    data_root="/kaggle/input",
    kaggle_inputs=True,           # one generator per attached dataset
    ood_generator=OOD_GEN,
    img_size=224,
    batch_size=64,
    head_epochs=3,                # Phase 1: frozen backbone
    finetune_epochs=5,            # Phase 2: unfreeze last blocks
    unfreeze_blocks=2,
    max_per_class_per_gen=4000,   # Kaggle has full subsets; raise vs Colab
    target_fpr=0.01,
    experiment="trustlens-deepfake",
    out_dir="/kaggle/working/models",
)
cfg

## 4. Train (two-phase) + evaluate ID vs OOD
MLflow logs to `/kaggle/working/mlruns` (downloadable from the Output tab).

In [ ]:
import mlflow
mlflow.set_tracking_uri("file:/kaggle/working/mlruns")
from trustlens.training.train import train
model, report, logs = train(cfg)

## 5. The distribution-shift exhibit

In [ ]:
import json, matplotlib.pyplot as plt
print(report.summary_line())
print(json.dumps(report.as_dict(), indent=2))

labels = ["ID test", f"OOD\n({report.ood_generator})"]
accs = [report.id_test.accuracy, report.ood.accuracy]
plt.bar(labels, accs, color=["#2a9d8f", "#e76f51"]); plt.ylim(0, 1)
plt.ylabel("accuracy")
plt.title(f"OOD accuracy drop = {report.accuracy_drop:.3f} ({report.relative_drop:.0%})")
for i, a in enumerate(accs): plt.text(i, a + 0.01, f"{a:.3f}", ha="center")
plt.show()

## 6. Inspect the MLflow run

In [ ]:
import mlflow
run = mlflow.search_runs(experiment_names=[cfg.experiment]).iloc[0]
print(run[[c for c in run.index if c.startswith("metrics.")]])

## 7. Save the model
Anything in `/kaggle/working` is downloadable from the notebook's **Output** tab, or **Save Version** to persist it.

In [ ]:
print("Best checkpoint:", "/kaggle/working/models/efficientnet_b0_best.pt")
import os; print(os.listdir('/kaggle/working/models'))

## Notes

- **Why hold a family out?** Fraudsters use *new* generators; in-distribution accuracy overstates real performance. The OOD drop is the number that matters.
- **Full subsets here:** on Kaggle the datasets mount without download, so raise `max_per_class_per_gen` (or set it to `None`) and/or `finetune_epochs` if you have GPU time — better headline numbers than the Colab validation-subset run.
- **Different hold-out:** set `OOD_GEN` (cell 2) to any name in `GENERATORS`.
- **Add training families:** just attach more GenImage generator datasets and rerun.